In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'Rate'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.25
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = False
_MC_INCLUDE_BY_PREDICTOR = True
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))


Known R assumption: True
Augmented measurement equation: Rate
Augmented coefficient: x_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[3.021 0.    0.   ]
 [0.    4.196 0.   ]
 [0.    0.    0.195]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.200,0.465,0.050,0.009,1000,72,0.072,0.008,0.058,0.090
1,Infl,1.680,0.398,0.066,0.010,1000,125,0.125,0.010,0.106,0.147
2,Rate,1.055,0.485,0.046,0.009,1000,56,0.056,0.007,0.043,0.072


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 1000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.121,3.253,0.484,0.002,0.091,0.009,1000,68,0.068,0.008,0.054,0.085,3.0,200,4
1,cov_identity,1.699,167.161,0.000,0.008,2.025,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,0.955,0.051,1.323,0.723,0.417,0.008,0.044,0.002,0.003,0.033,0.010,0.0,1000,118,0.118,0.010,0.099,0.139
1,OutGap,x,-0.377,-0.074,0.337,-1.057,0.359,0.010,0.011,0.002,0.001,0.030,0.009,0.0,1000,180,0.180,0.012,0.157,0.205
2,OutGap,r,-1.502,-0.039,2.629,-0.551,0.446,0.007,0.088,0.002,0.010,0.033,0.010,0.0,1000,96,0.096,0.009,0.079,0.116
3,Infl,Pi,-0.777,-0.033,1.574,-0.472,0.451,0.006,0.051,0.002,0.003,0.032,0.009,0.0,1000,79,0.079,0.009,0.064,0.097
4,Infl,x,0.063,0.012,0.402,0.167,0.481,0.006,0.013,0.002,0.002,0.033,0.009,0.0,1000,56,0.056,0.007,0.043,0.072
5,Infl,r,-0.487,-0.008,3.130,-0.115,0.487,0.005,0.104,0.002,0.012,0.033,0.009,0.0,1000,56,0.056,0.007,0.043,0.072
6,Rate,Pi,-0.067,-0.015,0.316,-0.217,0.475,0.006,0.010,0.002,0.001,0.033,0.009,0.0,1000,64,0.064,0.008,0.050,0.081
7,Rate,x,0.047,0.039,0.081,0.560,0.432,0.007,0.003,0.002,0.000,0.033,0.009,0.0,1000,98,0.098,0.009,0.081,0.118
8,Rate,r,-0.162,-0.016,0.627,-0.220,0.476,0.006,0.021,0.002,0.002,0.033,0.009,0.0,1000,47,0.047,0.007,0.036,0.062


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,0.203,0.016,0.944,0.228,0.515,0.005,0.028,0.002,0.001,0.030,0.009,0.0,1000,45,0.045,0.007,0.034,0.060
1,OutGap,x,-0.140,-0.037,0.241,-0.520,0.510,0.005,0.007,0.002,0.001,0.027,0.009,0.0,1000,61,0.061,0.008,0.048,0.078
0,OutGap,r,-1.196,-0.034,2.472,-0.482,0.474,0.006,0.076,0.002,0.009,0.031,0.009,0.0,1000,70,0.070,0.008,0.056,0.088
5,Infl,Pi,-0.398,-0.024,1.122,-0.335,0.485,0.006,0.036,0.002,0.001,0.032,0.009,0.0,1000,65,0.065,0.008,0.051,0.082
4,Infl,x,-0.039,-0.006,0.287,-0.090,0.503,0.005,0.009,0.002,0.001,0.032,0.009,0.0,1000,56,0.056,0.007,0.043,0.072
3,Infl,r,0.071,0.003,2.942,0.037,0.497,0.005,0.093,0.002,0.010,0.032,0.009,0.0,1000,46,0.046,0.007,0.035,0.061
8,Rate,Pi,0.049,0.015,0.225,0.206,0.502,0.005,0.007,0.002,0.000,0.032,0.009,0.0,1000,52,0.052,0.007,0.040,0.068
7,Rate,x,0.033,0.038,0.058,0.536,0.446,0.007,0.002,0.002,0.000,0.033,0.009,0.0,1000,86,0.086,0.009,0.070,0.105
6,Rate,r,-0.188,-0.019,0.589,-0.270,0.486,0.005,0.019,0.002,0.002,0.032,0.009,0.0,1000,51,0.051,0.007,0.039,0.066


In [11]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.741,-0.786,0.955,0.955,0.0,0.0,0.0,0.032,0.020,0.044,0.044,0.0,0.0,0.0
1,OutGap,x,-0.009,-0.368,-0.377,-0.377,0.0,0.0,0.0,0.008,0.006,0.011,0.011,0.0,0.0,0.0
2,OutGap,r,-0.069,-1.433,-1.502,-1.502,0.0,0.0,0.0,0.063,0.049,0.088,0.088,0.0,0.0,0.0
3,Infl,Pi,-0.127,-0.650,-0.777,-0.777,0.0,0.0,0.0,0.023,0.046,0.051,0.051,0.0,0.0,0.0
4,Infl,x,0.016,0.047,0.063,0.063,0.0,0.0,0.0,0.006,0.012,0.013,0.013,0.0,0.0,0.0
5,Infl,r,-0.181,-0.305,-0.487,-0.487,0.0,0.0,0.0,0.047,0.098,0.104,0.104,0.0,0.0,0.0
6,Rate,Pi,0.008,-0.075,-0.067,-0.067,0.0,0.0,0.0,0.005,0.010,0.010,0.010,0.0,0.0,0.0
7,Rate,x,-0.001,0.048,0.047,0.047,0.0,0.0,0.0,0.001,0.003,0.003,0.003,0.0,0.0,0.0
8,Rate,r,-0.057,-0.105,-0.162,-0.162,-0.0,0.0,0.0,0.010,0.020,0.021,0.021,0.0,0.0,0.0


In [12]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.794,-1.592,0.203,0.203,0.0,0.0,0.0,0.022,0.011,0.028,0.028,0.0,0.0,0.0
1,OutGap,x,0.316,-0.456,-0.140,-0.140,-0.0,0.0,0.0,0.005,0.004,0.007,0.007,0.0,0.0,0.0
2,OutGap,r,-1.269,0.073,-1.196,-1.196,0.0,0.0,0.0,0.067,0.040,0.076,0.076,0.0,0.0,0.0
3,Infl,Pi,-0.043,-0.355,-0.398,-0.398,-0.0,0.0,0.0,0.016,0.032,0.036,0.036,0.0,0.0,0.0
4,Infl,x,-0.005,-0.033,-0.039,-0.039,-0.0,0.0,0.0,0.004,0.009,0.009,0.009,0.0,0.0,0.0
5,Infl,r,-0.090,0.160,0.071,0.071,-0.0,0.0,0.0,0.044,0.090,0.093,0.093,0.0,0.0,0.0
6,Rate,Pi,0.011,0.038,0.049,0.049,0.0,0.0,0.0,0.004,0.006,0.007,0.007,0.0,0.0,0.0
7,Rate,x,0.002,0.031,0.033,0.033,0.0,0.0,0.0,0.001,0.002,0.002,0.002,0.0,0.0,0.0
8,Rate,r,-0.046,-0.143,-0.188,-0.188,-0.0,0.0,0.0,0.009,0.018,0.019,0.019,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.

In [13]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()


## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [14]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,0.049,-1330.65,-1328.767,3.767,0.214,0.001,0.895,0.897,0.111,0.008,1000,397,0.397,0.015,0.367,0.428


In [15]:
res_mle

OptimizationResult(kind='mle', x=array([0.05938171]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(0.05938170733623821), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1290.0894598385466), loglik=np.float64(-1290.0894598385466), logprior=np.float64(0.0), logpost=np.float64(-1290.0894598385466), nfev=12, nit=4, raw=  message: CONVERGENC

## Serial Autocorrelation Tests for the Augmented Model

In [16]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.168,0.471,0.049,0.009,1000,70,0.070,0.008,0.056,0.088
1,Infl,1.692,0.397,0.067,0.010,1000,127,0.127,0.011,0.108,0.149
2,Rate,1.045,0.484,0.045,0.009,1000,50,0.050,0.007,0.038,0.065


In [17]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.121,3.253,0.484,0.002,0.091,0.009,1000,68,0.068,0.008,0.054,0.085,3.0,200,4
1,cov_identity,1.699,167.161,0.000,0.008,2.025,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.121,3.240,0.485,0.002,0.090,0.009,1000,69,0.069,0.008,0.055,0.086,3.0,200,4
1,cov_identity,1.690,153.774,0.000,0.008,1.868,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.121,0.002,0.121,0.002,0.000,0.000,3.253,0.091,3.240,0.090,0.013,0.004,0.507,0.016,0.476,0.538
1,cov_identity,1000,1.699,0.008,1.690,0.008,0.009,0.001,167.161,2.025,153.774,1.868,13.387,0.436,0.615,0.015,0.584,0.645
